In [ ]:
import json
import re
from collections.abc import Callable
from pathlib import Path

TICKERS = ("C", "JPM", "WFC")
SHORT_TABLE_CHARS = 120

BULLET_PATTERN = re.compile(
    r"(^|\n)\s*(?:[\u2022\u25aa\u25e6\u2023-])\s+"
)


def find_project_root() -> Path:
    current_path = Path.cwd().resolve()

    for candidate in (current_path, *current_path.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate

    raise FileNotFoundError(
        "Nije pronađen pyproject.toml. Sačuvaj notebook unutar projekta."
    )


def load_tables(project_root: Path, ticker: str) -> list[dict]:
    path = (
        project_root
        / "data"
        / "processed"
        / "elements"
        / f"{ticker.lower()}.jsonl"
    )

    records = [
        json.loads(line)
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

    return [
        add_table_metrics(record)
        for record in records
        if record["element_type"] == "table"
    ]


def add_table_metrics(record: dict) -> dict:
    text = str(record["text"]).strip()
    rows = [row.strip() for row in text.splitlines() if row.strip()]
    plain_text = " ".join(text.split())

    content_columns = max(
        (
            sum(bool(cell.strip()) for cell in row.split(" | "))
            for row in rows
        ),
        default=0,
    )

    word_count = len(re.findall(r"\b[\w'-]+\b", plain_text))
    has_numeric_value = bool(re.search(r"\d", plain_text))
    has_sentence_end = bool(re.search(r"[.!?](?:\s|$)", plain_text))

    return {
        **record,
        "row_count": len(rows),
        "content_columns": content_columns,
        "char_count": len(plain_text),
        "word_count": word_count,
        "has_numeric_value": has_numeric_value,
        "bullet_like": bool(BULLET_PATTERN.search(text)),
        "title_like": (
            len(rows) <= 2
            and content_columns <= 1
            and len(plain_text) <= SHORT_TABLE_CHARS
            and word_count <= 15
            and not has_numeric_value
            and not plain_text.endswith((".", "!", "?", ";"))
        ),
        "paragraph_like": (
            content_columns <= 1
            and len(plain_text) > SHORT_TABLE_CHARS
            and has_sentence_end
        ),
    }


def shorten(text: str, limit: int = 220) -> str:
    text = " ".join(text.split())
    return text if len(text) <= limit else f"{text[:limit].rstrip()}..."


project_root = find_project_root()
tables_by_ticker = {
    ticker: load_tables(project_root, ticker)
    for ticker in TICKERS
}

categories: dict[str, Callable[[dict], bool]] = {
    "one_row": lambda table: table["row_count"] == 1,
    "one_content_column": lambda table: table["content_columns"] <= 1,
    "very_short": lambda table: table["char_count"] <= SHORT_TABLE_CHARS,
    "no_numeric_values": lambda table: not table["has_numeric_value"],
    "title_like": lambda table: table["title_like"],
    "bullet_like": lambda table: table["bullet_like"],
    "paragraph_like": lambda table: table["paragraph_like"],
}

print(f"Project root: {project_root}")
print("\nCOUNTS")

for ticker, tables in tables_by_ticker.items():
    print(f"\n{ticker}: tables={len(tables)}")

    for category, matches in categories.items():
        count = sum(matches(table) for table in tables)
        print(f"  {category:<20} {count:>5}")


def select_examples(
    category: str,
    limit: int = 8,
) -> list[tuple[str, dict]]:
    matches = categories[category]
    pools = {
        ticker: sorted(
            (table for table in tables if matches(table)),
            key=lambda table: table["char_count"],
            reverse=True,
        )
        for ticker, tables in tables_by_ticker.items()
    }

    examples: list[tuple[str, dict]] = []

    while len(examples) < limit:
        added = False

        for ticker in TICKERS:
            if pools[ticker] and len(examples) < limit:
                examples.append((ticker, pools[ticker].pop(0)))
                added = True

        if not added:
            break

    return examples


print("\nEXAMPLES")

example_categories = (
    "one_row",
    "one_content_column",
    "title_like",
    "bullet_like",
    "paragraph_like",
)

for category in example_categories:
    print(f"\n[{category}]")

    examples = select_examples(category)

    if not examples:
        print("  Nema primera.")
        continue

    for ticker, table in examples:
        flags = [
            name
            for name, matches in categories.items()
            if matches(table)
        ]

        print(
            f"  {ticker} order={table['order_index']} "
            f"rows={table['row_count']} "
            f"cols={table['content_columns']} "
            f"chars={table['char_count']} "
            f"flags={','.join(flags)}"
        )
        print(f"    {shorten(str(table['text']))}")

Project root: C:\Users\nikola.bakic\OneDrive - Sixsentix AG\Documents\Repositories\Banking-Technology-and-Operational-Risk-Intelligence-Assistant

COUNTS

C: tables=326
  one_row                 14
  one_content_column       7
  very_short              18
  no_numeric_values       13
  title_like               7
  bullet_like              0
  paragraph_like           0

JPM: tables=678
  one_row                368
  one_content_column      41
  very_short             366
  no_numeric_values       40
  title_like              32
  bullet_like              3
  paragraph_like           8

WFC: tables=17
  one_row                  1
  one_content_column       1
  very_short               2
  no_numeric_values        2
  title_like               0
  bullet_like              0
  paragraph_like           1

EXAMPLES

[one_row]
  C order=3921 rows=1 cols=3 chars=985 flags=one_row
    Titi Cole Former Head of Legacy Franchises, Citigroup Inc. Ellen M. Costello Chair, Citibank, N.A. Grace E. Dai

In [ ]:
import importlib

import bankscope.parsing.sec_html_parser as sec_html_parser

importlib.reload(sec_html_parser)
normalize_text = sec_html_parser.normalize_text

In [ ]:

synthetic_cases = {
    "\ufeffOperational risk": "Operational risk",
    "cyber\u200bsecurity\u2060 risk": "cybersecurity risk",
    "inter\u00adnational operations": "international operations",
    "non\u00a0breaking\u202fspaces": "non breaking spaces",
}

for original, expected in synthetic_cases.items():
    result = normalize_text(original)
    assert result == expected, (repr(original), repr(result), repr(expected))

preserved_cases = (
    "U.S. GAAP legal‑entity net income",
    "Yes ¨ No þ",
    "Risk exposure: $1,234.50",
)

for original in preserved_cases:
    result = normalize_text(original)
    assert result == original, (repr(original), repr(result))

print("Synthetic Unicode checks passed.")

print("Synthetic Unicode checks passed.")

for ticker in ("c", "jpm", "wfc"):
    path = project_root / "data" / "processed" / "elements" / f"{ticker}.jsonl"
    changes = []

    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue

        record = json.loads(line)
        text = str(record["text"])

        old_normalized = " ".join(text.split())
        new_normalized = normalize_text(text)

        if old_normalized != new_normalized:
            changes.append(
                (
                    record["order_index"],
                    old_normalized,
                    new_normalized,
                )
            )

    print(f"\n{ticker.upper()}: unicode_changes={len(changes)}")

    for order_index, before, after in changes[:5]:
        print(f"  order={order_index}")
        print(f"    before: {before[:180]!r}")
        print(f"    after:  {after[:180]!r}")

Synthetic Unicode checks passed.
Synthetic Unicode checks passed.

C: unicode_changes=0

JPM: unicode_changes=0

WFC: unicode_changes=0


In [ ]:
import importlib
import json
from collections import Counter
from pathlib import Path

import bankscope.parsing.sec_html_parser as sec_html_parser

importlib.reload(sec_html_parser)

for ticker in ("c", "jpm", "wfc"):
    path = project_root / "data" / "processed" / "elements" / f"{ticker}.jsonl"

    records = [
        json.loads(line)
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

    converted = []

    for record in records:
        if record["element_type"] != "table":
            continue

        new_type = sec_html_parser.classify_table_element(record["text"])

        if new_type != "table":
            converted.append(
                {
                    "order_index": record["order_index"],
                    "new_type": new_type,
                    "text": record["text"],
                }
            )

    counts = Counter(record["new_type"] for record in converted)

    print(f"\n{ticker.upper()}: converted={len(converted)} {dict(counts)}")

    for new_type in ("heading", "list", "paragraph"):
        examples = [
            record for record in converted
            if record["new_type"] == new_type
        ][:3]

        for record in examples:
            text = " ".join(record["text"].split())
            print(
                f"  {new_type:<9} order={record['order_index']} "
                f"{text[:180]!r}"
            )


C: converted=7 {'heading': 7}
  heading   order=1054 'U.S. Personal Banking'
  heading   order=1059 'Branded Cards—Credit Cards'
  heading   order=1063 'Retail Services'

JPM: converted=40 {'heading': 32, 'paragraph': 7, 'list': 1}
  heading   order=934 'INTRODUCTION'
  heading   order=940 'EXECUTIVE OVERVIEW'
  heading   order=1006 'CONSOLIDATED RESULTS OF OPERATIONS'
  list      order=1355 'Asset Management has two high-level measures of its overall fund performance. • Percentage of active mutual fund and active ETF assets under management in funds rated 4- or 5-star:'
  paragraph order=1173 'Calculation of certain U.S. GAAP and non-GAAP financial measures Certain U.S. GAAP and non-GAAP financial measures are calculated as follows: Book value per share (“BVPS”) Common s'
  paragraph order=1218 'Consumer & Community Banking offers products and services to consumers and small businesses through bank branches, ATMs, digital (including mobile and online) and telephone banking'
  paragra

In [11]:
importlib.reload(sec_html_parser)

manifest_path = project_root / "artifacts" / "manifests" / "filings.json"
filings = json.loads(manifest_path.read_text(encoding="utf-8"))

for filing in filings:
    ticker = filing["ticker"]

    if ticker not in {"C", "JPM", "WFC"}:
        continue

    html_path = Path(filing["local_html_path"])
    if not html_path.is_absolute():
        html_path = project_root / html_path
    elements = sec_html_parser.parse_filing_html(html_path)
    navigation = [
        element
        for element in elements
        if element["is_navigation"]
    ]

    type_counts = {}

    for element in navigation:
        element_type = str(element["element_type"])
        type_counts[element_type] = type_counts.get(element_type, 0) + 1

    print(
        f"\n{ticker}: navigation={len(navigation)} "
        f"types={type_counts}"
    )

    for element in navigation[:10]:
        text = " ".join(str(element["text"]).split())

        print(
            f"  order={element['order_index']} "
            f"type={element['element_type']} "
            f"{text[:220]!r}"
        )


JPM: navigation=2 types={'table': 1, 'paragraph': 1}
  order=29 type=table 'Part I | | Page Item 1. | Business . | 1 | Overview | 1 | Business segments & Corporate | 1 | Competition | 1 | Supervision and regulation | 2-6 | Human capital | 7-8 | Distribution of assets, liabilities and stockholder'
  order=909 type=paragraph 'Table of contents'

C: navigation=1 types={'paragraph': 1}
  order=33 type=paragraph 'FORM 10-K CROSS-REFERENCE INDEX'

WFC: navigation=0 types={}


In [12]:
from bs4 import BeautifulSoup
from bs4.element import Tag

HEADING_TAGS = ("h1", "h2", "h3", "h4", "h5", "h6")
CENTER_PATTERN = re.compile(r"text-align\s*:\s*center", re.IGNORECASE)


def heading_signals(tag: Tag, text: str) -> list[str]:
    signals = []

    if tag.name in HEADING_TAGS:
        signals.append("heading_tag")

    style = str(tag.get("style", ""))
    align = str(tag.get("align", "")).casefold()

    if align == "center" or CENTER_PATTERN.search(style):
        signals.append("centered")

    bold_tag = tag.find(["b", "strong"])

    if (
        bold_tag is not None
        and sec_html_parser.normalize_text(
            bold_tag.get_text(" ", strip=True)
        )
        == text
    ):
        signals.append("fully_bold")

    letters = [character for character in text if character.isalpha()]

    if (
        len(letters) >= 4
        and all(character.isupper() for character in letters)
    ):
        signals.append("uppercase")

    return signals


for filing in filings:
    ticker = filing["ticker"]

    if ticker not in {"C", "JPM", "WFC"}:
        continue

    html_path = Path(filing["local_html_path"])

    if not html_path.is_absolute():
        html_path = project_root / html_path

    soup = BeautifulSoup(html_path.read_bytes(), "lxml")

    for tag in soup(["script", "style", "noscript", "ix:header"]):
        tag.decompose()

    for tag in soup.find_all(style=sec_html_parser.HIDDEN_STYLE_PATTERN):
        tag.decompose()

    root = soup.body or soup
    candidates = []

    for tag in root.find_all(
        ("div", "p", "h1", "h2", "h3", "h4", "h5", "h6")
    ):
        if tag.find_parent("table") is not None:
            continue

        text = sec_html_parser.normalize_text(
            tag.get_text(" ", strip=True)
        )

        if (
            not text
            or len(text) > 180
            or len(sec_html_parser.WORD_PATTERN.findall(text)) > 24
            or text.endswith((".", "!", "?", ";"))
        ):
            continue

        signals = heading_signals(tag, text)

        if signals:
            candidates.append((tag.name, signals, text))

    signal_counts = {}

    for _, signals, _ in candidates:
        for signal in signals:
            signal_counts[signal] = signal_counts.get(signal, 0) + 1

    print(
        f"\n{ticker}: candidates={len(candidates)} "
        f"signals={signal_counts}"
    )

    for tag_name, signals, text in candidates[:20]:
        print(
            f"  tag={tag_name:<3} "
            f"signals={','.join(signals):<30} "
            f"{text[:180]!r}"
        )

C:\Users\nikola.bakic\AppData\Local\Temp\ipykernel_20248\1008938807.py:53: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html_path.read_bytes(), "lxml")



JPM: candidates=348 signals={'centered': 316, 'uppercase': 37}
  tag=div signals=centered,uppercase             'UNITED STATES'
  tag=div signals=centered,uppercase             'SECURITIES AND EXCHANGE COMMISSION'
  tag=div signals=centered,uppercase             'WASHINGTON, D.C. 20549'
  tag=div signals=centered,uppercase             'FORM 10-K'
  tag=div signals=centered                       'Annual report pursuant to Section 13 or 15(d) of'
  tag=div signals=centered                       'the Securities Exchange Act of 1934'
  tag=div signals=centered                       'For the fiscal year ended Commission file December 31 , 2025 number 1-5805'
  tag=div signals=centered                       '(Exact name of registrant as specified in its charter)'
  tag=div signals=centered                       'Registrant’s telephone number, including area code: ( 212 ) 270-6000'
  tag=div signals=centered                       'Securities registered pursuant to Section 12(b) of the Act:'
